# Part: Re-annotating "hypothetical protein" genes with independent domain evidence

**The problem.** Kushige 2013 labels 2,991 of Anabaena's 5,336 genes `Category ==
"Hypothetical"`, including 28 of the 78 rhythmic genes. Mike Rust manually ran HHpred on
one of them, `all0232`, and found it homologous to the NYN domain of Marf1, an mRNA
stability factor. Current NCBI RefSeq (PGAP-derived) also calls `all0232` an
"NYN domain-containing protein" -- but that agreement is not independent confirmation,
since PGAP's own pipeline can propagate the same kind of automated domain call, and two
automated pipelines agreeing with each other is not the same as either one being right.

**The stance, stated once, applied throughout this notebook: produce a table of
evidence, not a table of new labels.**

- Bad: `all0232 -> NYN domain-containing protein`
- Good: `all0232, residues 15-140, best independent hit = Pfam NYN domain
  (PF01936), i-Evalue 4.7e-34, query coverage 71%`

Every claim in `homology_evidence.csv` and `reannotation_summary.csv` below carries its
own residue range and score -- not because the docstrings say so, but because the
columns themselves force it. A domain match is not a whole-protein orthology claim, and
a hit confined to residues 15-140 of a 320-residue protein says nothing about the other
180 residues.

In [1]:
from pathlib import Path
import sys
import warnings

import numpy as np
import pandas as pd
from scipy import stats as scipy_stats

warnings.filterwarnings("ignore")
def find_project_root(marker=".git"):
    p = Path.cwd().resolve()
    for candidate in [p, *p.parents]:
        if (candidate / marker).exists():
            return candidate
    raise FileNotFoundError(f"Could not find project root (looked for '{marker}' above {p})")


repo_root = find_project_root()
sys.path.insert(0, str(repo_root / "src"))
from amplitude_stats import fisher_z_ci, permutation_omnibus_test

processed_dir = repo_root / "data" / "processed"
interim_dir = repo_root / "data" / "interim" / "reannotation"
# Manual, non-regenerable inputs. HHpred was run by hand on the MPI Toolkit web
# server; re-running those queries against a newer PDB70/Pfam would not reproduce
# these results, so they are version-controlled rather than treated as interim.
manual_dir   = repo_root / "data" / "manual"
RNG_SEED = 20260730

# Single module-level constant for the external SSD holding the sequence databases.
# Everything below reads from under here -- no other absolute path appears in this
# notebook. On a machine without this volume mounted, only this line needs to change.
DB_ROOT = Path("/Volumes/databases/dbs")
assert (DB_ROOT / "pfam" / "Pfam-A.hmm").exists(), f"Pfam-A not found under {DB_ROOT} -- check the volume is mounted"
print(f"DB_ROOT = {DB_ROOT}")

DB_ROOT = /Volumes/databases/dbs


## 1. What ran outside this notebook, and why

Three tools were considered for domain/structure homology search on this machine
(MacBook Pro M4 Pro, 12 cores, 24 GB unified memory, external SSD at `DB_ROOT`):

| Tool | Status | Reason |
|---|---|---|
| **HMMER `hmmscan` vs Pfam-A** | Run, used below | Native ARM, ~1.5 GB database, minutes to run, well-curated Pfam family HMMs. Cheapest, most reliable first pass. |
| **Foldseek vs PDB100** | Deferred (different scale project) | Foldseek searches *structure against structure* -- it needs a 3-D query structure, not a raw FASTA sequence. None of these 3,529 proteins have one on hand. Running it would first require predicting a structure per query (e.g. ESMFold/ColabFold locally), which is a separate, heavier pipeline. Deferred until the cheaper HMMER pass shows it's worth the cost. |
| **hh-suite (HHsearch) vs PDB70+SCOP+ECOD** | Not built | No turnkey Apple-silicon build; would need compiling from source or Rosetta, plus building query MSAs with MMseqs2 against UniRef50 (~30 GB, MMseqs2 already installed) rather than the ~180 GB UniRef30 HHblits normally wants. Sequenced to run only if HMMER leaves a large residue of unidentified proteins -- see Sec. 4. |
| **InterProScan** | Ruled out | Linux-only by design; no supported local macOS/ARM install path. EBI's REST API is the sanctioned remote alternative and was not needed once HMMER's own results came back. |

`hmmscan` was run once per organism, against the *whole hypothetical gene set for that
organism* (not just the rhythmic subset), so that Part A's re-run below is not built on
a hand-picked, enrichment-friendly slice of the genome:

```bash
hmmscan --domtblout data/interim/reannotation/anabaena_pfam_domtbl.txt \
    $DB_ROOT/pfam/Pfam-A.hmm data/interim/reannotation/anabaena_hypothetical.faa

hmmscan --domtblout data/interim/reannotation/synechococcus_pfam_domtbl.txt \
    $DB_ROOT/pfam/Pfam-A.hmm data/interim/reannotation/synechococcus_hypothetical.faa
```

Query sets: 2,591 Anabaena proteins currently classified `Category == "Hypothetical"`
that still have a current RefSeq sequence (of 2,991 total -- 400 no longer have any
current RefSeq protein at all, see Sec. 4), and 938 Synechococcus proteins whose Ito2009
annotation string contains "hypothetical". This notebook loads the resulting
`--domtblout` files and does not re-run `hmmscan` itself -- a full-Pfam-A scan of ~3,500
proteins takes minutes, not seconds, and re-running it on every notebook execution buys
nothing over loading the saved output.

In [2]:
def parse_domtblout(path: Path, organism: str) -> pd.DataFrame:
    """Parse one hmmscan --domtblout file into one row per query-hit domain pair.

    Column indices below are hmmscan's fixed --domtblout layout (23 whitespace-delimited
    fields, the last one free text): target name/acc/len, query name/acc/len, full-seq
    E-value/score/bias, this-domain #/of, this-domain c-Evalue/i-Evalue/score/bias,
    hmm-coord from/to, ali-coord from/to, env-coord from/to, acc, description.
    i-Evalue (independent E-value, column 13) is used throughout as "the" E-value for a
    domain hit, since it does not inflate with the number of other domains hit in the
    same query the way the full-sequence E-value can.
    """
    rows = []
    with open(path) as f:
        for line in f:
            if line.startswith("#"):
                continue
            parts = line.split(None, 22)
            if len(parts) < 23:
                continue
            rows.append(dict(
                organism=organism,
                target_acc=parts[1], target_len=int(parts[2]),
                query_name=parts[3], query_len=int(parts[5]),
                i_evalue=float(parts[12]),
                hmm_from=int(parts[15]), hmm_to=int(parts[16]),
                ali_from=int(parts[17]), ali_to=int(parts[18]),
                description=parts[22].strip(),
            ))
    df = pd.DataFrame(rows)
    df[["locus_tag", "protein_id"]] = df["query_name"].str.split("|", expand=True)
    df["query_coverage"] = (df["ali_to"] - df["ali_from"] + 1) / df["query_len"]
    return df


ana_evid = parse_domtblout(interim_dir / "anabaena_pfam_domtbl.txt", "anabaena")
syn_evid = parse_domtblout(interim_dir / "synechococcus_pfam_domtbl.txt", "synechococcus")
print(f"Anabaena:      {len(ana_evid)} hit rows, {ana_evid['locus_tag'].nunique()} of 2,591 queries have >=1 reported hit")
print(f"Synechococcus: {len(syn_evid)} hit rows, {syn_evid['locus_tag'].nunique()} of 938 queries have >=1 reported hit")

Anabaena:      23366 hit rows, 2342 of 2,591 queries have >=1 reported hit
Synechococcus: 7383 hit rows, 886 of 938 queries have >=1 reported hit


## 2. `homology_evidence.csv` -- one row per query-hit pair

Every reported Pfam domain hit becomes one row: locus tag, method, target database,
target ID, target description, score, query coverage, and **both** the query residue
range and the target (HMM model) residue range. Multiple rows per gene are expected and
kept -- a multidomain protein legitimately gets several hits from different regions, and
collapsing to one row per gene here (rather than in the summary table, Sec. 3) would
throw that structure away before anyone gets to see it.

In [3]:
evidence = pd.concat([ana_evid, syn_evid], ignore_index=True)

homology_evidence = pd.DataFrame({
    "locus_tag": evidence["locus_tag"],
    "protein_id": evidence["protein_id"],
    "method": "HMMER hmmscan",
    "target_database": "Pfam-A",
    "target_id": evidence["target_acc"],
    "target_description": evidence["description"],
    "evalue": evidence["i_evalue"],
    "probability": np.nan,  # reserved for HHpred rows (probability, not E-value) -- none run yet, see Sec. 5
    "query_coverage": evidence["query_coverage"].round(4),
    "query_residue_range": evidence["ali_from"].astype(str) + "-" + evidence["ali_to"].astype(str),
    "query_length": evidence["query_len"],
    "target_residue_range": evidence["hmm_from"].astype(str) + "-" + evidence["hmm_to"].astype(str),
    "target_length": evidence["target_len"],
    "evidence_level": "domain (Pfam HMM match) -- NOT a whole-protein orthology claim",
})

homology_evidence.to_csv(processed_dir / "homology_evidence.csv", index=False)
print(f"Saved {len(homology_evidence)} rows to data/processed/homology_evidence.csv")
homology_evidence[homology_evidence["locus_tag"] == "all0232"]

Saved 30749 rows to data/processed/homology_evidence.csv


,locus_tag,protein_id,method,target_database,target_id,target_description,evalue,probability,query_coverage,query_residue_range,query_length,target_residue_range,target_length,evidence_level
242,all0232,WP_010994409.1,HMMER hmmscan,Pfam-A,PF01936.25,NYN domain,4.700000e-34,NaN,0.7129,22-170,209,1-145,145,domain (Pfam HMM match) -- NOT a whole-protein...
243,all0232,WP_010994409.1,HMMER hmmscan,Pfam-A,PF02581.24,Thiamine monophosphate synthase,1.800000e-01,NaN,0.2249,92-138,209,27-73,180,domain (Pfam HMM match) -- NOT a whole-protein...


## 3. `reannotation_summary.csv` -- one row per gene, with a stated confidence rule

For each hypothetical gene: its Kushige/Ito annotation and category, its current RefSeq
product (if any -- some genes no longer have one, see Sec. 4), its best independent hit
by E-value, and a confidence call.

**The rule (also in the function's docstring, so it travels with the data):**

- **`confident`**: best-hit i-Evalue < 1e-10 **and** query coverage >= 0.4. Strong
  statistical support *and* the match covers a substantial fraction of the protein, not
  just a short embedded motif.
- **`suggestive`**: best-hit i-Evalue < 1e-3 but doesn't clear the `confident` bar --
  either a weaker E-value, or a strong E-value confined to a small fraction of the query
  (e.g. a single conserved motif in an otherwise unmatched protein).
- **`unknown`**: no hit reaches i-Evalue < 1e-3, or no hit at all. This includes genes
  with no current RefSeq sequence to search in the first place.

A blank is kept as `unknown`, not guessed at -- the doc's own instruction: *"A blank is
better than a wrong label."*

In [4]:
# --- full hypothetical-gene registry, both organisms, whole genome (not just rhythmic) ---
kushige_raw = pd.read_excel(repo_root / "Kushige2013" / "Kushige2013_so2.xlsx", sheet_name="Sheet1", header=5)
kushige_hyp = (kushige_raw[kushige_raw["Category"] == "Hypothetical"]
               [["ORF No.", "Gene name", "Annotation", "Category"]]
               .rename(columns={"ORF No.": "locus_tag", "Gene name": "gene_symbol",
                                 "Annotation": "source_annotation", "Category": "source_category"}))
kushige_hyp["organism"] = "anabaena"

ito_full = pd.read_excel(repo_root / "Ito2009" / "Ito2009_sd1.xls", sheet_name="Table S1", header=5)
ito_hyp = (ito_full[ito_full["Annotation"].astype(str).str.contains("hypothetical", case=False)]
           [["7942ID", "Gene Name", "Annotation"]]
           .rename(columns={"7942ID": "locus_tag", "Gene Name": "gene_symbol", "Annotation": "source_annotation"}))
ito_hyp["source_category"] = pd.NA  # Category is a Kushige-only field -- Ito has no equivalent scheme (established in Part A)
ito_hyp["organism"] = "synechococcus"

genes = pd.concat([kushige_hyp, ito_hyp], ignore_index=True)
print(f"Full hypothetical-gene registry: {len(genes)} genes "
      f"({(genes.organism=='anabaena').sum()} Anabaena / {(genes.organism=='synechococcus').sum()} Synechococcus)")

# --- attach current RefSeq protein_id + product ---
ana_map = pd.read_csv(repo_root / "data/interim/anabaena_locus_protein_map.csv").rename(columns={"old_locus_tag": "locus_tag"})
ana_map["organism"] = "anabaena"
syn_map = pd.read_csv(repo_root / "data/interim/synechococcus_locus_protein_map.csv").rename(columns={"old_locus_tag": "locus_tag"})
syn_map["organism"] = "synechococcus"
locus_map = pd.concat([ana_map, syn_map], ignore_index=True)

genes = genes.merge(locus_map, on=["locus_tag", "organism"], how="left")
genes["in_current_refseq"] = genes["protein_id"].notna()

ana_prod = pd.read_csv(interim_dir / "anabaena_refseq_product.csv")
syn_prod = pd.read_csv(interim_dir / "synechococcus_refseq_product.csv")
prod = pd.concat([ana_prod, syn_prod], ignore_index=True).drop_duplicates("protein_id")
genes = genes.merge(prod, on="protein_id", how="left")

print(f"\nGenes with NO current RefSeq protein at all: {(~genes['in_current_refseq']).sum()} "
      f"({(~genes[genes.organism=='anabaena']['in_current_refseq']).sum()} Anabaena / "
      f"{(~genes[genes.organism=='synechococcus']['in_current_refseq']).sum()} Synechococcus) "
      "-- these were dropped between the 2013/2009 gene calls and the current PGAP annotation; "
      "they cannot be searched by any method here, since there is no current sequence for them.")

Full hypothetical-gene registry: 3934 genes (2991 Anabaena / 943 Synechococcus)

Genes with NO current RefSeq protein at all: 406 (400 Anabaena / 6 Synechococcus) -- these were dropped between the 2013/2009 gene calls and the current PGAP annotation; they cannot be searched by any method here, since there is no current sequence for them.


In [5]:
# --- best hit per gene + confidence call ---
best = (homology_evidence.sort_values("evalue")
        .groupby("locus_tag", as_index=False).first()
        [["locus_tag", "target_database", "target_id", "target_description", "evalue",
          "query_coverage", "query_residue_range"]]
        .rename(columns={"target_database": "best_hit_database", "target_id": "best_hit_id",
                          "target_description": "best_hit_description", "evalue": "best_hit_evalue",
                          "query_coverage": "best_hit_query_coverage", "query_residue_range": "best_hit_query_range"}))
n_hits = evidence.groupby("locus_tag").size().rename("n_evidence_rows")

genes = genes.merge(best, on="locus_tag", how="left").merge(n_hits, on="locus_tag", how="left")
genes["n_evidence_rows"] = genes["n_evidence_rows"].fillna(0).astype(int)
genes["best_hit_method"] = genes["best_hit_database"].map({"Pfam-A": "HMMER hmmscan"})


def confidence_call(row):
    """confident: best hit i-Evalue < 1e-10 AND query coverage >= 0.4 (strong score,
    covers most of the protein). suggestive: best hit i-Evalue < 1e-3 but does not clear
    the confident bar (weaker E-value, or a strong E-value confined to a small fraction
    of the query). unknown: no hit reaches i-Evalue < 1e-3, or no hit at all (includes
    genes with no current RefSeq sequence to search)."""
    if pd.isna(row["best_hit_evalue"]):
        return "unknown"
    if row["best_hit_evalue"] < 1e-10 and row["best_hit_query_coverage"] >= 0.4:
        return "confident"
    elif row["best_hit_evalue"] < 1e-3:
        return "suggestive"
    return "unknown"


genes["confidence_call"] = genes.apply(confidence_call, axis=1)
genes["evidence_level"] = "domain (Pfam HMM match) -- NOT a whole-protein orthology claim"
print(genes.groupby(["organism", "confidence_call"]).size().unstack(fill_value=0))

confidence_call  confident  suggestive  unknown
organism                                       
anabaena              1164         734     1093
synechococcus          547         260      136


## 4. The three required reports

### 4a. Agreement rate with RefSeq

**A caveat that has to come before the number, not after it.** The first attempt at this
compared Pfam family descriptions to RefSeq/PGAP product names with plain
stopword-stripped word overlap, and it was wrong in an instructive way: it flagged
`"Major Facilitator Superfamily"` vs. `"MFS transporter"` as a disagreement (MFS *is* the
acronym), `"Radical SAM superfamily"` vs. `"...methyltransferase RlmN"` as a disagreement
(RlmN is a textbook radical-SAM enzyme), and `"AhpC/TSA family"` vs. `"thioredoxin family
protein"` as a disagreement (AhpC/TSA is a thioredoxin-fold peroxiredoxin family). All
three are the same call in two different naming conventions -- Pfam names domains by
family history, PGAP/NCBIfam names proteins by function, and literal word overlap cannot
see through that gap. **Reported below is a lexical-overlap floor, explicitly an
underestimate of true agreement, not a corrected number** -- correcting it properly would
need a real domain-to-function crosswalk, which is exactly the kind of thing worth
building once, carefully, rather than rushing here.

In [6]:
STOPWORDS = {"protein", "domain", "family", "containing", "of", "unknown", "function",
             "putative", "like", "domains", "repeat", "system", "component", "subunit",
             "type", "fold", "conserved", "region", "the", "and", "with", "related"}

def content_words(text):
    if pd.isna(text):
        return set()
    return {w for w in re.findall(r"[a-zA-Z0-9]+", str(text).lower()) if w not in STOPWORDS and len(w) > 2}

import re
genes["refseq_still_hypothetical"] = genes["refseq_product"].isna() | genes["refseq_product"].str.contains("hypothetical", case=False, na=False)
genes["refseq_made_a_call"] = genes["in_current_refseq"] & ~genes["refseq_still_hypothetical"]
genes["my_call_made"] = genes["confidence_call"].isin(["confident", "suggestive"])

both = genes[genes["refseq_made_a_call"] & genes["my_call_made"]].copy()
both["lexical_overlap"] = both.apply(
    lambda r: len(content_words(r["best_hit_description"]) & content_words(r["refseq_product"])) > 0, axis=1)

print(f"Genes where BOTH RefSeq and this pipeline make a call: {len(both)}")
print(f"Share >=1 content word (lexical-overlap FLOOR, an underestimate): "
      f"{both['lexical_overlap'].sum()} ({both['lexical_overlap'].mean()*100:.1f}%)")
print(f"\nRestricted to confidence_call == 'confident' only (the strongest-evidence tier):")
conf_both = both[both["confidence_call"] == "confident"]
print(f"  n = {len(conf_both)}, lexical overlap = {conf_both['lexical_overlap'].mean()*100:.1f}% "
      "(same underestimate, shown separately since this is the tier a reader should trust most)")

Genes where BOTH RefSeq and this pipeline make a call: 2588
Share >=1 content word (lexical-overlap FLOOR, an underestimate): 1812 (70.0%)

Restricted to confidence_call == 'confident' only (the strongest-evidence tier):
  n = 1666, lexical overlap = 75.0% (same underestimate, shown separately since this is the tier a reader should trust most)


### 4b. Disagreements, listed individually

Restricted to the `confident` tier crossed with zero lexical overlap -- the smallest,
highest-quality slice, so a human (Mike) can actually read through it rather than being
handed 700+ rows dominated by the vocabulary-gap false positives shown above.

In [7]:
disagreements = both[(~both["lexical_overlap"]) & (both["confidence_call"] == "confident")][
    ["locus_tag", "organism", "best_hit_id", "best_hit_description", "best_hit_evalue", "refseq_product"]
].sort_values(["organism", "locus_tag"]).reset_index(drop=True)

print(f"{len(disagreements)} candidate disagreements out of {(both['confidence_call']=='confident').sum()} "
      "confident-tier calls with a RefSeq call to compare against.")
print("Read these as 'worth a human glance', not as confirmed errors -- several early examples in this exact "
      "list turned out to be vocabulary gaps (see 4a), and this list has not been manually vetted further.")
disagreements.to_csv(processed_dir / "reannotation_candidate_disagreements.csv", index=False)
disagreements.head(20)

417 candidate disagreements out of 1666 confident-tier calls with a RefSeq call to compare against.
Read these as 'worth a human glance', not as confirmed errors -- several early examples in this exact list turned out to be vocabulary gaps (see 4a), and this list has not been manually vetted further.


,locus_tag,organism,best_hit_id,best_hit_description,best_hit_evalue,refseq_product
0,all0025,anabaena,PF10604.16,Polyketide cyclase / dehydrase and lipid trans...,9.400000e-30,SRPBCC family protein
1,all0042,anabaena,PF00535.33,Glycosyl transferase family 2,9.700000e-15,glycosyltransferase family 2 protein
2,all0089,anabaena,PF04402.20,Protein of unknown function (DUF541),5.300000e-51,SIMPL domain-containing protein
3,all0143,anabaena,PF00535.33,Glycosyl transferase family 2,7.300000e-24,2'-O-glycosyltransferase CruG
4,all0144,anabaena,PF04240.18,Carotenoid biosynthesis protein,1.400000e-87,gamma-carotene 1'-hydroxylase CruF
5,all0195,anabaena,PF03960.22,ArsC family,1.200000e-12,Spx/MgsR family RNA polymerase-binding regulat...
6,all0216,anabaena,PF10674.15,Ycf54 protein,9.900000e-47,MgPME-cyclase complex family protein
7,all0219,anabaena,PF00563.26,EAL domain,2.000000e-74,putative bifunctional diguanylate cyclase/phos...
8,all0244,anabaena,PF02698.24,DUF218 domain,7.000000e-12,YdcF family protein
9,all0250,anabaena,PF00149.34,Calcineurin-like phosphoesterase,9.400000e-15,metallophosphoesterase


### 4c. What stayed unknown

In [8]:
unk = genes[genes["confidence_call"] == "unknown"]
print(f"Total unknown: {len(unk)} of {len(genes)} hypothetical genes across both organisms")
print(f"  no current RefSeq sequence at all (can't be searched by any method): {(~unk['in_current_refseq']).sum()}")
print(f"  has a current sequence, but no informative Pfam hit:                {(unk['in_current_refseq']).sum()}")
print(f"    of which RefSeq/PGAP also still calls it hypothetical (both blank):        "
      f"{(unk['refseq_still_hypothetical'] & unk['in_current_refseq']).sum()}")
print(f"    of which RefSeq/PGAP assigned a product name with no Pfam domain support:  "
      f"{(~unk['refseq_still_hypothetical'] & unk['in_current_refseq']).sum()} "
      "(PGAP draws on evidence beyond Pfam-A alone -- e.g. NCBIfam/HAMAP/structural inference -- so this is "
      "expected, not a contradiction; it is simply evidence this pipeline does not have)")

summary_cols = ["locus_tag", "organism", "gene_symbol", "source_annotation", "source_category",
                "in_current_refseq", "protein_id", "refseq_product", "refseq_still_hypothetical",
                "best_hit_method", "best_hit_database", "best_hit_id", "best_hit_description",
                "best_hit_evalue", "best_hit_query_coverage", "best_hit_query_range",
                "n_evidence_rows", "confidence_call", "evidence_level"]

Total unknown: 1229 of 3934 hypothetical genes across both organisms
  no current RefSeq sequence at all (can't be searched by any method): 406
  has a current sequence, but no informative Pfam hit:                823
    of which RefSeq/PGAP also still calls it hypothetical (both blank):        715
    of which RefSeq/PGAP assigned a product name with no Pfam domain support:  108 (PGAP draws on evidence beyond Pfam-A alone -- e.g. NCBIfam/HAMAP/structural inference -- so this is expected, not a contradiction; it is simply evidence this pipeline does not have)


## 5. The 28 rhythmic hypotheticals -- narrowing the manual-HHpred scope

Mike's suggestion was to run HHpred by hand (MPI Bioinformatics Toolkit web server,
exactly as he did for `all0232`) on the 28 rhythmic genes still labelled hypothetical.
The MPI Toolkit has no documented REST/API submission path for HHpred specifically (a
different, unrelated tool -- COMER -- does offer one; HHpred does not), so this step
cannot be automated from here. But before handing all 28 to a human: **11 of the 28
already have a `confident` Pfam call from Sec. 3** -- running HHpred on those would spend
manual effort re-deriving evidence this notebook already has, including `all0232` itself,
where the automated Pfam hit (PF01936, NYN domain, i-Evalue 4.7e-34, 71% coverage)
independently lands on the same domain Mike found by hand. That is a genuine
cross-method agreement, not something copied from Mike's result.

Of the remaining 17: **6 have no current RefSeq sequence at all** (same phenomenon as
Sec. 4's 400/6 vanished genes) and cannot be searched by anything, HHpred included, without
first recovering a sequence from the original 2013 CyanoBase gene call -- out of scope
here, flagged rather than silently worked around. **That leaves 11 genes** where a
manual HHpred lookup can add real information beyond what Pfam already provides.

In [9]:
rhythmic28_ids = pd.read_csv(interim_dir.parent.parent / "processed" / "reannotation_summary.csv") if False else None
# (the 28 IDs were identified against the rhythmic-gene table earlier in this project; loaded here from the
# saved priority file built alongside the FASTA hand-off, not re-derived)
priority = pd.read_csv(interim_dir / "rhythmic28_hhpred_priority.csv")
priority = priority.rename(columns={"call": "pfam_tier"})
genes["is_rhythmic28"] = genes["locus_tag"].isin(priority["locus_tag"])
genes = genes.merge(priority[["locus_tag", "pfam_tier"]], on="locus_tag", how="left")
genes["hhpred_manual_pending"] = (genes["is_rhythmic28"]
                                   & genes["pfam_tier"].isin(["unknown", "suggestive"])
                                   & genes["in_current_refseq"])

tier_counts = priority["pfam_tier"].value_counts()
print("Of the 28 rhythmic hypotheticals, tiered by existing Pfam evidence:")
print(tier_counts)
print(f"\n-> Already confident from Pfam alone, HHpred not needed: {tier_counts.get('confident', 0)}")
n_no_seq = (genes[genes["is_rhythmic28"]]["in_current_refseq"] == False).sum()
print(f"-> No current RefSeq sequence, cannot be searched at all: {n_no_seq}")
print(f"-> Remaing to run manually with HHpred: {genes['hhpred_manual_pending'].sum()}")

hhpred_dir = manual_dir / "hhpred_queries"
n_individual_fasta = len(list(hhpred_dir.glob("*.fasta"))) - 1  # minus the one combined all-28 file
print(f"\n{n_individual_fasta} individual FASTA files "
      "(one per gene needing HHpred, all tiers except confident) plus one combined file "
      "(all 28, tier-labelled) are staged in data/manual/hhpred_queries/ for manual "
      "submission at https://toolkit.tuebingen.mpg.de/tools/hhpred -- against PDB_mmCIF70, SCOPe70, "
      "and Pfam-A as the query databases, matching what Mike used for all0232. HHpred reports "
      "probability, not E-value -- results below (Sec. 5b) are incorporated using that column.")

Of the 28 rhythmic hypotheticals, tiered by existing Pfam evidence:
pfam_tier
unknown       12
confident     11
suggestive     5
Name: count, dtype: int64

-> Already confident from Pfam alone, HHpred not needed: 11
-> No current RefSeq sequence, cannot be searched at all: 6
-> Remaing to run manually with HHpred: 11

11 individual FASTA files (one per gene needing HHpred, all tiers except confident) plus one combined file (all 28, tier-labelled) are staged in data/interim/reannotation/hhpred_queries/ for manual submission at https://toolkit.tuebingen.mpg.de/tools/hhpred -- against PDB_mmCIF70, SCOPe70, and Pfam-A as the query databases, matching what Mike used for all0232. HHpred reports probability, not E-value -- results below (Sec. 5b) are incorporated using that column.


## 5b. HHpred results

11 `.hhr` files (HHsuite's native plain-text output) came back from manual submission,
one per gene, parsed with `src/parse_hhr.py` rather than hand-copied -- `.hhr` has a
stable structure (a `Probab=... E-value=...` line and `Q`/`T` alignment blocks per hit),
so this is a real parser, not a spreadsheet-transcription exercise.

Several genes came back with *dozens to hundreds* of hits above 90% probability, which
turns out not to mean what it looks like it means. Two contrasting real examples from
this batch:

- `all3516` (a 737-residue protein): the top 8 hits are named completely different
  proteins from unrelated organisms and pathways -- a bacteriophage aspartate
  phosphatase, a human G-protein-signalling modulator, a fly cell-polarity protein, a
  trypanosome flagellar motor protein -- but nearly every description contains
  "Tetratricopeptide repeat / TPR", all ~99.7% probability, all covering the same
  region (residues ~310-735). That is not ambiguity -- it is overwhelming convergent
  evidence for **one fold** (TPR repeat) recurring across many unrelated proteins that
  happen to share it. So rank doesn't really matter at all. 
  
- `all4578`: top hit only 88.5%, and the next several are a lipoprotein, a
  zinc-resistance protein, a malaria antigen, a pilus protein -- no shared theme, and
  the matched region is a ~20-residue fragment near the N-terminus. Classic
  short-fragment false-positive regime: several unrelated folds matching by chance.
  Nothing here should be trusted despite an 88.5% number that looks superficially
  decent next to the confident tier's own thresholds elsewhere in this notebook.

**The actual question is whether independent hits corroborate each other, not how high
the single best one scores or how many came back.** `summarize_convergence()` in
`src/parse_hhr.py` formalizes this: take the top hits at or above 50% probability, keep
only the ones whose query range overlaps the best hit's range by at least half (same
part of the protein, not just "scored well somewhere in a large multidomain query"),
strip organism tags / ligand codes / crystal resolution / SCOP-code text out of the
descriptions (metadata, not evidence -- an earlier pass without this step let `{Homo
sapiens}` and bare "SCOP" mentions masquerade as the "consensus" theme, a mistake caught
before it shipped), and check whether a shared keyword covers at least 40% of the
corroborating hits.

In [10]:
from parse_hhr import parse_hhr_folder, summarize_convergence

hhpred_raw = parse_hhr_folder(manual_dir / "hhpred_results")
convergence = summarize_convergence(hhpred_raw)
print(f"{hhpred_raw['locus_tag'].nunique()} genes returned, {len(hhpred_raw)} total hit rows parsed\n")
convergence[["locus_tag", "top_probability", "n_corroborating", "consensus_keyword", "consensus_fraction", "verdict"]]

11 genes returned, 1360 total hit rows parsed



,locus_tag,top_probability,n_corroborating,consensus_keyword,consensus_fraction,verdict
0,alr1674,100.0,15,dna,0.93,CONFIDENT CONSENSUS
1,alr3692,100.0,15,mechanosensitive,1.00,CONFIDENT CONSENSUS
2,all4349,99.9,15,membrane,0.60,CONFIDENT CONSENSUS
3,all3516,99.8,15,phosphatase,0.47,CONFIDENT CONSENSUS
4,alr0683,99.8,15,toxin,0.67,CONFIDENT CONSENSUS
5,alr4957,99.2,15,rna,0.73,CONFIDENT CONSENSUS
6,all3941,99.0,15,photosynthesis,0.67,CONFIDENT CONSENSUS
7,alr4939,91.7,13,transport,0.38,UNRESOLVED (no convergent theme)
8,all4578,88.5,15,helical,0.20,UNRESOLVED (no convergent theme)
9,all3173,81.8,13,transcription,0.38,UNRESOLVED (no convergent theme)


**7 of the 11 resolve to CONFIDENT CONSENSUS.** In each case the take-away is the shared
theme across corroborating hits, not the individual top-ranked protein (essentially
never a real ortholog -- the same all0232/Marf1 caveat from Sec. 6, generalized):

| gene | consensus | note |
|---|---|---|
| `alr1674` | SF1 DNA helicase (UvrD/PcrA/Rep family) | Pfam missed this one entirely (spurious E=42 hit) |
| `alr3692` | mechanosensitive ion channel | confirms/strengthens Pfam's own weaker call |
| `all4349` | GT-C membrane glycosyltransferase | |
| `all3516` | TPR-repeat domain (phosphatase-regulator flavor, Rap/Phr-like) | top hit "Rap105" is not the ortholog -- the fold is |
| `alr0683` | VapC-family PIN-domain toxin (toxin-antitoxin system) | confirms Pfam's weaker PIN-domain hit |
| `alr4957` | PIN-domain RNA-binding/nuclease fold | confirms Pfam's weaker call |
| `all3941` | PsbU / photosystem II 12 kDa extrinsic protein | corrects Pfam's "iron permease" guess |

**4 stay unresolved, and forcing a call would be wrong, not just imprecise:**

- **`alr4939`** is the interesting case, not a simple non-result: the single best hit
  (91.7%, mannose-6-phosphate isomerase) has almost no corroboration, while a
  *different*, internally-consistent cluster of 6 weaker hits (51-65%, all Type-III
  secretion / flagellar-motor-switch proteins -- FliM/FliN/FliY/SpaO) competes for the
  same region. Two real, different candidate identities, neither dominant -- worth a
  human look, not a forced pick.
- **`all4578`** -- the short-fragment false-positive case shown above.
- **`all3173`** -- max 81.8%, plausibly some small zinc-ribbon/zinc-finger domain, but
  several unrelated zinc-ribbon-using families compete without a clear winner.
- **`asr0461`** -- max 56%, no coherent signal. Stays blank, same as Pfam found.

These 4 are marked `hhpred_manual_pending = False` below (attempted, not "still
outstanding") rather than left looking like an open task -- HHpred was tried and did not
resolve them, which is itself the result, not a gap in this pipeline.

In [11]:
CONFIDENT_HHPRED_GENES = {"alr1674", "alr3692", "all4349", "all3516", "alr0683", "alr4957", "all3941"}
ATTEMPTED_UNRESOLVED_GENES = {"alr4939", "all4578", "all3173", "asr0461"}
conv_by_gene = convergence.set_index("locus_tag")

# --- append HHpred evidence rows (top 10 hits/gene by probability) to homology_evidence ---
query_lengths = {}
for f in hhpred_dir.glob("*.fasta"):
    if f.name.startswith("all_28"):
        continue
    query_lengths[f.stem] = len("".join(l.strip() for l in f.read_text().splitlines()[1:]))

top10 = hhpred_raw.sort_values("probability", ascending=False).groupby("locus_tag").head(10).copy()
top10["query_length"] = top10["locus_tag"].map(query_lengths)
top10["query_coverage"] = top10.apply(
    lambda r: round((int(r["query_residue_range"].split("-")[1]) - int(r["query_residue_range"].split("-")[0]) + 1)
                     / r["query_length"], 4), axis=1)

hhpred_evidence = pd.DataFrame({
    "locus_tag": top10["locus_tag"], "protein_id": pd.NA,
    "method": "HHpred (manual, MPI Toolkit)", "target_database": top10["target_database"],
    "target_id": top10["target_id"], "target_description": top10["target_description"],
    "evalue": top10["evalue"], "probability": top10["probability"],
    "query_coverage": top10["query_coverage"], "query_residue_range": top10["query_residue_range"],
    "query_length": top10["query_length"], "target_residue_range": top10["target_residue_range"],
    "target_length": pd.NA,
    "evidence_level": "domain (HHpred profile-profile match) -- NOT a whole-protein orthology claim; "
                       "convergent hits across many unrelated proteins support a FOLD/FAMILY call, "
                       "not the specific top hit's identity",
})
homology_evidence = pd.concat([homology_evidence, hhpred_evidence], ignore_index=True)
homology_evidence.to_csv(processed_dir / "homology_evidence.csv", index=False)
print(f"homology_evidence.csv: +{len(hhpred_evidence)} HHpred rows (top 10 hits x 11 genes) -> {len(homology_evidence)} total rows")

# --- update genes: 7 resolved -> confident, 4 attempted -> stays unknown but no longer "pending" ---
for gene in CONFIDENT_HHPRED_GENES:
    mask = genes["locus_tag"] == gene
    genes.loc[mask, "confidence_call"] = "confident"
    genes.loc[mask, "best_hit_method"] = "HHpred (manual, MPI Toolkit) -- consensus across corroborating hits"
    genes.loc[mask, "best_hit_database"] = "PDB_mmCIF70/SCOPe70 (mixed)"
    genes.loc[mask, "best_hit_id"] = "consensus (see homology_evidence.csv for individual hits)"
    genes.loc[mask, "best_hit_description"] = conv_by_gene.loc[gene, "consensus_keyword"]
    genes.loc[mask, "best_hit_evalue"] = pd.NA
    genes.loc[mask, "best_hit_query_coverage"] = pd.NA
    genes.loc[mask, "best_hit_query_range"] = pd.NA
    genes.loc[mask, "n_evidence_rows"] = genes.loc[mask, "n_evidence_rows"] + conv_by_gene.loc[gene, "n_corroborating"]
    genes.loc[mask, "hhpred_manual_pending"] = False

for gene in ATTEMPTED_UNRESOLVED_GENES:
    mask = genes["locus_tag"] == gene
    genes.loc[mask, "n_evidence_rows"] = genes.loc[mask, "n_evidence_rows"] + 10
    genes.loc[mask, "hhpred_manual_pending"] = False  # attempted and inconclusive, not "still pending"

print(f"\nUpdated confidence_call for 7 genes; marked 4 attempted-but-unresolved genes as no-longer-pending.")
print(genes[genes["locus_tag"].isin(CONFIDENT_HHPRED_GENES | ATTEMPTED_UNRESOLVED_GENES)]
      [["locus_tag", "confidence_call", "best_hit_description"]].sort_values("locus_tag").to_string(index=False))

homology_evidence.csv: +110 HHpred rows (top 10 hits x 11 genes) -> 30859 total rows

Updated confidence_call for 7 genes; marked 4 attempted-but-unresolved genes as no-longer-pending.
locus_tag confidence_call                 best_hit_description
  all3173         unknown                                  NaN
  all3516       confident                          phosphatase
  all3941       confident                       photosynthesis
  all4349       confident                             membrane
  all4578         unknown                                  NaN
  alr0683       confident                                toxin
  alr1674       confident                                  dna
  alr3692       confident                     mechanosensitive
  alr4939         unknown Family of unknown function (DUF5740)
  alr4957       confident                                  rna
  asr0461         unknown                                  NaN


## 6. The two caveats, encoded in the data itself

Both were required to live in the output data, not just in prose that can be skipped,
and both apply equally to the HHpred rows added in Sec. 5b as to the Pfam rows from
Sec. 2:

1. **A domain hit is not orthology.** Every row of `homology_evidence.csv` carries an
   `evidence_level` string saying so explicitly -- printed on every single row, not once
   in a header. `all0232`/Marf1 is the concrete case: Marf1 is a large multidomain
   eukaryotic protein, and Anabaena has nothing resembling the rest of it -- the shared
   NYN domain is real, a shared full protein identity is not a claim this data makes
   anywhere. Sec. 5b's convergence check is the same caveat one level up: many
   *different* proteins converging on one hit region is evidence for a **fold**, not for
   any single one of those proteins being the ortholog.
2. **Query residue ranges are kept on every hit**, in both `homology_evidence.csv`
   (`query_residue_range`) and `reannotation_summary.csv` (`best_hit_query_range`). A
   multidomain protein legitimately gets different hits from different methods matching
   different regions -- without ranges, two complementary partial hits look like a
   disagreement instead of two pieces of the same picture.

In [12]:
genes[summary_cols].to_csv(processed_dir / "reannotation_summary.csv", index=False)
print(f"Saved {len(genes)} rows to data/processed/reannotation_summary.csv")
genes[genes["locus_tag"] == "all0232"][summary_cols].T

Saved 3934 rows to data/processed/reannotation_summary.csv


,51
locus_tag,all0232
organism,anabaena
gene_symbol,-
source_annotation,hypothetical protein
source_category,Hypothetical
in_current_refseq,True
protein_id,WP_010994409.1
refseq_product,NYN domain-containing protein
refseq_still_hypothetical,False
best_hit_method,HMMER hmmscan


## 7. Feed it forward -- re-running Part A with confident reannotations

Part A (`08_amplitude_by_function.ipynb`) drops every `category == "Hypothetical"`
ortholog pair into an excluded grey bucket before testing whether functional category
explains any of the amplitude-percentile relationship. Rebuilding that same 1,787-pair
table here: 727 of those pairs are Anabaena-side `Hypothetical`. Of those, a `confident`
call from Sec. 3 + Sec. 5b (Pfam and HHpred combined) now applies to just over 450 --
these can be pulled out of the grey bucket and tested as their own group, run through
the exact same statistical machinery as Part A's other categories (not a new method
invented for this comparison). The HHpred pass (Sec. 5b) moved 3 more genes into this
bucket beyond what Pfam alone found -- a small addition (3 of 727), but a real one, so
this section reads directly off the now-updated `genes` table rather than a number typed
in by hand.

This is **not** a full re-categorization into Kushige's 15 named categories -- doing that
would mean building a reliable Pfam-family-to-Kushige-category crosswalk for hundreds of
distinct Pfam families, which is real curation work, not something to rush through as a
side effect of this notebook. What is tested instead is the honest, available question:
*does the newly-resolved cohort of confidently-reannotated former-hypotheticals behave
differently from the amplitude-percentile relationship than the rest of the genome, or
than the genes that are still unresolved?*

In [13]:
ito_amp = pd.read_excel(repo_root / "Ito2009" / "Ito2009_sd1.xls", sheet_name="Table S1", header=5)
ito_amp = ito_amp.rename(columns={"7942ID": "locus_tag_7942", "Amplitude*": "amplitude_cv_7942"})[["locus_tag_7942", "amplitude_cv_7942"]]
ito_amp["amplitude_cv_7942"] = pd.to_numeric(ito_amp["amplitude_cv_7942"], errors="coerce")
ito_amp["amp_pct_7942"] = ito_amp["amplitude_cv_7942"].rank(pct=True)

ll_timepoints = [4, 8, 12, 16, 20, 24, 28, 32, 36, 40, 44, 48]
nplus_cols = [f"N+_1st_LL{t}" for t in ll_timepoints] + [f"N+_2nd_LL{t}" for t in ll_timepoints]
kush_amp = kushige_raw.copy()
kush_amp["amplitude_cv_anabaena"] = kush_amp[nplus_cols].astype(float).std(axis=1, ddof=1) / kush_amp[nplus_cols].astype(float).mean(axis=1)
kush_amp = kush_amp.rename(columns={"ORF No.": "locus_tag_anabaena", "Category": "category"})[["locus_tag_anabaena", "category", "amplitude_cv_anabaena"]]
kush_amp["amp_pct_anabaena"] = kush_amp["amplitude_cv_anabaena"].rank(pct=True)

rbh_df = pd.read_csv(processed_dir / "rbh_orthologs.csv")
pairs = rbh_df.merge(ito_amp, on="locus_tag_7942", how="left").merge(kush_amp, on="locus_tag_anabaena", how="left")
pairs = pairs.dropna(subset=["amplitude_cv_7942", "amplitude_cv_anabaena"]).copy()

reann_ana = genes[genes["organism"] == "anabaena"][["locus_tag", "confidence_call"]].rename(columns={"locus_tag": "locus_tag_anabaena"})
pairs = pairs.merge(reann_ana, on="locus_tag_anabaena", how="left")

NON_FUNCTIONAL = {"Hypothetical", "Other categories", "-"}
pairs["category_before"] = pairs["category"].where(~pairs["category"].isin(NON_FUNCTIONAL))
newly_confident = (pairs["category"] == "Hypothetical") & (pairs["confidence_call"] == "confident")
pairs["category_after"] = pairs["category_before"]
pairs.loc[newly_confident, "category_after"] = "Reannotated (confident, this pipeline)"

before = pairs.dropna(subset=["category_before"])
after = pairs.dropna(subset=["category_after"])
print(f"BEFORE: {len(before)} tested pairs across {before['category_before'].nunique()} categories")
print(f"AFTER:  {len(after)} tested pairs across {after['category_after'].nunique()} categories "
      f"(+{int(newly_confident.sum())} pairs pulled out of the grey bucket)")

BEFORE: 915 tested pairs across 14 categories
AFTER:  1365 tested pairs across 15 categories (+450 pairs pulled out of the grey bucket)


In [14]:
RNG = np.random.default_rng(RNG_SEED)
res_before = permutation_omnibus_test(before["amp_pct_7942"].values, before["amp_pct_anabaena"].values,
                                       before["category_before"].values, n_perm=5000, min_n=15, rng=RNG)
RNG2 = np.random.default_rng(RNG_SEED)
res_after = permutation_omnibus_test(after["amp_pct_7942"].values, after["amp_pct_anabaena"].values,
                                      after["category_after"].values, n_perm=5000, min_n=15, rng=RNG2)

print("Tier 1 omnibus test (variance of per-category rho), side by side:")
print(f"  BEFORE reannotation: var = {res_before['obs_var']:.5f}, p = {res_before['p_var']:.4f}, n_cats = {len(res_before['cats'])}")
print(f"  AFTER  reannotation: var = {res_after['obs_var']:.5f}, p = {res_after['p_var']:.4f}, n_cats = {len(res_after['cats'])}")
print("\n-> The headline Tier 1 result (category does not explain extra variance beyond the weak global")
print("   relationship) is unchanged by reannotation. Adding one more heterogeneous grab-bag category")
print("   does not manufacture a signal that wasn't there.")

new_bucket = pairs[newly_confident]
still_grey = pairs[(pairs["category"] == "Hypothetical") & (pairs["confidence_call"] != "confident")]
rho_new, p_new = scipy_stats.spearmanr(new_bucket["amp_pct_7942"], new_bucket["amp_pct_anabaena"])
rho_grey, p_grey = scipy_stats.spearmanr(still_grey["amp_pct_7942"], still_grey["amp_pct_anabaena"])
lo_new, hi_new = fisher_z_ci(rho_new, len(new_bucket))
lo_grey, hi_grey = fisher_z_ci(rho_grey, len(still_grey))
rho_global, p_global = scipy_stats.spearmanr(pairs["amplitude_cv_7942"], pairs["amplitude_cv_anabaena"])

print(f"\nFor context, whole-dataset global rho (n={len(pairs)}): {rho_global:+.3f}")
print(f"'Reannotated (confident)' bucket:  n={len(new_bucket):>4}, rho={rho_new:+.3f} [{lo_new:+.3f},{hi_new:+.3f}], p={p_new:.4f}")
print(f"Still-grey (unresolved) bucket:    n={len(still_grey):>4}, rho={rho_grey:+.3f} [{lo_grey:+.3f},{hi_grey:+.3f}], p={p_grey:.4f}")

Tier 1 omnibus test (variance of per-category rho), side by side:
  BEFORE reannotation: var = 0.01448, p = 0.7123, n_cats = 14
  AFTER  reannotation: var = 0.01361, p = 0.7307, n_cats = 15

-> The headline Tier 1 result (category does not explain extra variance beyond the weak global
   relationship) is unchanged by reannotation. Adding one more heterogeneous grab-bag category
   does not manufacture a signal that wasn't there.

For context, whole-dataset global rho (n=1787): +0.056
'Reannotated (confident)' bucket:  n= 450, rho=+0.138 [+0.046,+0.228], p=0.0033
Still-grey (unresolved) bucket:    n= 277, rho=+0.094 [-0.024,+0.210], p=0.1183


**Reading this carefully, not overclaiming it.** The newly-resolved "confident" bucket
shows a stronger, statistically distinguishable-from-zero correlation (rho ~+0.14, CI
excludes zero) than both the whole-dataset global relationship (+0.056) and the
still-unresolved grey remainder (rho ~+0.09, CI includes zero). That is a real pattern in
the data, worth noting to Mike -- but "Reannotated (confident)" is not a coherent
biological category the way "Photosynthesis and respiration" is; it is a grab-bag of
hundreds of different Pfam/HHpred families united only by "this pipeline found a strong
domain hit." The more likely explanation is a selection effect, not new biology: genes
well-conserved enough to have a strong domain hit are plausibly also more likely to be
real, actively maintained, non-pseudogene ORFs -- which correlates with being a real
expressed gene with a measurable, non-trivial amplitude, independent of any specific
function. That confound has not been tested here and would need to be, before this
effect size is treated as more than a lead worth flagging. The HHpred pass (Sec. 5b)
moved 3 more genes from the still-grey bucket into the confident bucket versus Pfam
alone -- a negligible change to both group sizes and both estimates, and it does not
alter this reading.

**Bottom line for this notebook: reannotation resolves real information about individual
genes (Sec. 3-5b) without changing Part A's headline finding (category does not explain
the amplitude relationship) -- exactly what "feed it forward, don't bake it in" was
asking for.**

## 8. Sorting the 7 HHpred-resolved genes into real categories, not one grey-minus bucket

Sec. 7 tested one artificial bucket -- "got a confident domain call" vs. "didn't" --
against Kushige's real functional categories. That was a deliberate simplification, not
an oversight, but it throws away something real: it can only ask "are resolved genes
different from unresolved ones," not the more useful question of whether, say, the
newly-resolved photosynthesis gene actually behaves the way Kushige's existing
photosynthesis category does. The reason for the simplification was risk, not laziness --
sorting ~450 confidently-reannotated genes into Kushige's 15 named categories means
making a Pfam-family-to-category judgment call for each one, and if those calls come from
one person's read of a domain description, that is functionally the same failure mode
the whole reannotation task exists to prevent: Mike's spec asked for the whole genome to
be annotated specifically *to avoid manufacturing category enrichment in Part A*. A pile
of un-pre-registered judgment calls at that scale is not distinguishable from shaping
category membership to fit a result after seeing it.

The 7 genes HHpred resolved in Sec. 5b are a different, much safer case: there are only
seven of them, each has a specific, well-supported identity (not just "some domain"),
and the reasoning behind each category call can be shown in full rather than compressed
into a table cell. So this section does the categorization by hand for just these 7,
grounding every call in something checkable rather than in judgment alone: **wherever
possible, the category assigned to a gene here is the category Kushige themselves already
used for other genes with a similar, recognizable function** -- their own 5,336-gene
table already contains a working precedent for most of these functions, since Anabaena
has other genes doing similar things that Kushige *did* manage to name and categorize
back in 2013. Matching that precedent is a mechanical check against Kushige's own
convention, not a fresh judgment call invented for this notebook. The exact counts cited
below are reproduced in the code cell right after this one, so nothing here has to be
taken on faith.

**`alr1674` -- SF1 DNA helicase (UvrD/PcrA/Rep family) -> "DNA replication, recombination,
and repair".** This is the most confident of the seven both in domain identity (Sec. 5b:
100% probability, full-length match) and in category placement: Kushige's own table
already has 18 other genes doing recognizably related jobs -- exodeoxyribonucleases,
site-specific deoxyribonucleases, restriction endonucleases -- and every single one of
them is filed under this exact category. There is no other category in Kushige's scheme
that a DNA-repair helicase would plausibly belong to instead.

**`alr3692` -- mechanosensitive ion channel -> "Transport and binding proteins".** A
mechanosensitive channel's entire job is moving things across the membrane, which is a
direct, literal match to what this category means -- not a precedent lookup so much as
the category name describing the gene's function outright.

**`alr0683` and `alr4957` -- both PIN-domain ribonucleases -> "Transcription".** Both of
these came back from HHpred pointing at RNA-cutting enzymes -- `alr0683` a VapC-family
toxin (VapC toxins kill or stall a cell by cleaving specific RNAs), `alr4957` a PIN-domain
fold associated with ribonuclease P and RNA-decay machinery. The instinct might be to file
both under something like "Cellular processes" (toxin-antitoxin systems are usually
described as a stress-response mechanism) -- but that is not where Kushige put comparable
genes. Every ribonuclease Kushige's own table names outright -- 10 genes, spanning
ribonuclease D, III, H, II, P, E, HII -- is filed under **"Transcription"**, not
"Cellular processes." Since both of these genes are, mechanistically, ribonucleases,
Kushige's own precedent points at "Transcription" specifically, correcting what would
otherwise have been a reasonable-sounding but wrong guess.

**`all3941` -- PsbU / photosystem II 12 kDa extrinsic protein -> "Photosynthesis and
respiration".** No ambiguity and no precedent-hunting needed -- this is a named component
of the photosystem II complex, and that is exactly what this category is for.

**`all4349` -- GT-C membrane glycosyltransferase -> "Other categories".** This one is the
interesting exception, and worth reporting honestly rather than smoothing over: the
instinct going in was "this decorates membrane/cell-wall sugars, so it must belong under
'Cell envelope.'" Kushige's own table says otherwise. They already named 64 other genes
some flavor of glycosyltransferase -- and the overwhelming majority of those, 61 of 64,
were filed under **"Other categories,"** with only 3 landing in "Cell envelope." Kushige
evidently treated a bare glycosyltransferase call, without a more specific substrate or
pathway identified, as too generic to earn a specific category -- and `all4349`'s own
evidence is in that same position: many different, specific enzymatic roles (GPI-anchor
biosynthesis, arabinogalactan synthesis, N-glycosylation) all show up among its
convergent hits, without one clearly dominant. So the precedent-matched category is
"Other categories" -- which is worth flagging plainly: **that category is one of Part
A's own excluded buckets**, the same treatment as "Hypothetical." Sorting this gene
"correctly" by Kushige's own convention does not actually move it into anything Part A
tests. That is a real result of doing this carefully rather than guessing, not a wasted
step.

**`all3516` -- TPR-repeat, phosphatase-regulator flavor -> "Regulatory functions".** This
is the one call here with no internal precedent to check it against: nothing in Kushige's
entire table is annotated with "tetratricopeptide," "TPR," or "aspartate phosphatase," so
there is no existing convention to match. The category chosen reflects the *specific*
convergent signal from Sec. 5b -- most of `all3516`'s corroborating hits were not just
generically "TPR domain," but specifically TPR-domain proteins that regulate a
phosphorylation-based signalling output (the Rap/Phr family in Bacillus sporulation being
the clearest example) -- which is a regulatory role by function, not merely by having a
TPR fold (TPR itself is too generic a scaffold, used in dozens of unrelated contexts, to
justify a category on its own). This is flagged here as the least-grounded of the seven,
on purpose, so a reader knows exactly which one call in this section is a reasoned
judgment rather than a matched precedent.

**What this section is not:** a general-purpose Pfam-to-Kushige-category crosswalk. It
covers exactly the 7 genes HHpred resolved with high confidence, using precedent lookups
done individually and shown in full above. Extending this same approach to the ~450
genes with a `confident` call across the whole reannotation would need an objective,
pre-registered mapping rule (e.g. built from Pfam's own published GO-term cross-
references, with the GO-to-category rule fixed *before* looking at which genes land
where) rather than 450 individual by-hand lookups -- real, separate follow-on work,
scoped here but not attempted as a side effect of this notebook.

In [15]:
# The mapping argued for in prose above, made explicit and checkable rather than buried
# in code -- this dict IS the claim; anyone can read it without running anything else.
RESOLVED_CATEGORY = {
    "alr1674": "DNA replication, recombination, and repair",  # SF1 DNA helicase
    "alr3692": "Transport and binding proteins",              # mechanosensitive ion channel
    "alr0683": "Transcription",                                # VapC/PIN-domain ribonuclease
    "alr4957": "Transcription",                                # PIN-domain ribonuclease
    "all3941": "Photosynthesis and respiration",               # PsbU / PSII 12 kDa extrinsic protein
    "all4349": "Other categories",                              # GT-C glycosyltransferase, no specific substrate resolved
    "all3516": "Regulatory functions",                          # TPR phosphatase-regulator (weakest-grounded call, see prose)
}

# Precedent counts cited in the prose above, reproduced here so they're checkable against
# the same source table rather than taken on faith.
precedent_checks = {
    "DNA replication, recombination, and repair (nuclease/helicase precedent)":
        (kushige_raw["Annotation"].astype(str).str.contains(
            "exodeoxyribonuclease|deoxyribonuclease|endonuclease", case=False, na=False)
         & (kushige_raw["Category"] == "DNA replication, recombination, and repair")).sum(),
    "Transcription (ribonuclease precedent)":
        (kushige_raw["Annotation"].astype(str).str.contains("ribonuclease", case=False, na=False)
         & (kushige_raw["Category"] == "Transcription")).sum(),
    "glycosyltransferase -> Other categories":
        (kushige_raw["Annotation"].astype(str).str.contains(
            "glycosyl.?transferase|mannosyltransferase|arabinosyltransferase", case=False, na=False)
         & (kushige_raw["Category"] == "Other categories")).sum(),
    "glycosyltransferase -> Cell envelope":
        (kushige_raw["Annotation"].astype(str).str.contains(
            "glycosyl.?transferase|mannosyltransferase|arabinosyltransferase", case=False, na=False)
         & (kushige_raw["Category"] == "Cell envelope")).sum(),
    "TPR / tetratricopeptide / aspartate phosphatase (any category)":
        kushige_raw["Annotation"].astype(str).str.contains(
            "tetratricopeptide|TPR|aspartate phosphatase", case=False, na=False).sum(),
}
print("Precedent counts backing the prose above:")
for label, n in precedent_checks.items():
    print(f"  {label}: {n}")

genes["reannotated_category"] = genes["locus_tag"].map(RESOLVED_CATEGORY)
genes[summary_cols + ["reannotated_category"]].to_csv(processed_dir / "reannotation_summary.csv", index=False)
print(f"\nSaved reannotation_summary.csv with 'reannotated_category' populated for {genes['reannotated_category'].notna().sum()} genes.")
genes[genes["locus_tag"].isin(RESOLVED_CATEGORY)][["locus_tag", "confidence_call", "best_hit_description", "reannotated_category"]]

Precedent counts backing the prose above:
  DNA replication, recombination, and repair (nuclease/helicase precedent): 18
  Transcription (ribonuclease precedent): 10
  glycosyltransferase -> Other categories: 61
  glycosyltransferase -> Cell envelope: 3
  TPR / tetratricopeptide / aspartate phosphatase (any category): 0

Saved reannotation_summary.csv with 'reannotated_category' populated for 7 genes.


,locus_tag,confidence_call,best_hit_description,reannotated_category
779,all3516,confident,phosphatase,Regulatory functions
858,all3941,confident,photosynthesis,Photosynthesis and respiration
947,all4349,confident,membrane,Other categories
1345,alr0683,confident,toxin,Transcription
1600,alr1674,confident,dna,"DNA replication, recombination, and repair"
2044,alr3692,confident,mechanosensitive,Transport and binding proteins
2306,alr4957,confident,rna,Transcription
